# Tugas Manipulasi Data dengan Pandas
Dataset: `data/paket_a_transaksi_pos_2024_2025.csv` (transaksi POS 2024–2025)

Isi: Task 1 Indexing & Selection · Task 2 Operasi Aritmatika · Task 3 Filtering · Task 4 Sorting · Task 5 Grouping & Agregasi

In [1]:
import pandas as pd

df = pd.read_csv('data/paket_a_transaksi_pos_2024_2025.csv')
print(df.shape)
df.head()

(5030, 9)


,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status
0,TRX0241065,2025-06-02,G09,K011,RMT1402,4,13000,0,SELESAI
1,TRX0240510,2024-12-28,G09,K015,SGR1205,1,21000,0,SELESAI
2,TRX0244483,2024-09-04,G11,K029,RMT1407,3,"7500,00",0,SELESAI
3,TRX0242532,2025-07-30,G05,K001,MKN1103,2,6500,0,SELESAI
4,TRX0243509,26/03/2025,G05,K016,RMT1407,1,6000,0,SELESAI


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5030 entries, 0 to 5029
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id_transaksi   5030 non-null   str  
 1   tanggal        5030 non-null   str  
 2   kode_gerai     5030 non-null   str  
 3   kode_kasir     4769 non-null   str  
 4   kode_produk    5030 non-null   str  
 5   jumlah         5030 non-null   int64
 6   harga_satuan   5030 non-null   str  
 7   diskon_persen  5030 non-null   int64
 8   status         5030 non-null   str  
dtypes: int64(2), str(7)
memory usage: 353.8 KB


# Task 1 — Indexing & Selection (Pembersihan Anomali & Imputasi)

### 1.1 Pembersihan format harga

In [3]:
# Kondisi awal: kolom bertipe object (string)
print(df['harga_satuan'].dtype)
print('Jumlah nilai berkoma  :', df['harga_satuan'].str.contains(',').sum())
print('Jumlah nilai bertitik :', df['harga_satuan'].str.contains(r'\.').sum())
df.loc[df['harga_satuan'].str.contains(r'[.,]'), 'harga_satuan'].drop_duplicates().head(10)

str
Jumlah nilai berkoma  : 51
Jumlah nilai bertitik : 45


2       7500,00
8      11000,00
19       13.000
51     27000,00
63     13500,00
77       28.000
84     43500,00
172    16500,00
237     4500,00
302      11.000
Name: harga_satuan, dtype: str

Selain format koma (`7500,00`), ternyata ada nilai seperti `13.000` / `50.500`. Dilihat dari kisaran harga produk (Rp4.000–Rp54.000), titik di sini adalah **pemisah ribuan** (13.000 = 13000), bukan desimal. Jika hanya mengganti koma menjadi titik, nilai ini akan terbaca `13.0` dan merusak perhitungan omzet. Maka titik ribuan dihapus dulu, baru koma diubah menjadi titik desimal.

In [4]:
h = df['harga_satuan'].astype(str).str.strip()
h = h.str.replace(r'^(\d{1,3})\.(\d{3})$', r'\1\2', regex=True)   # titik ribuan -> hapus
h = h.str.replace(',', '.', regex=False)                          # koma desimal -> titik
df['harga_satuan'] = h.astype(float)

print(df['harga_satuan'].dtype)
df['harga_satuan'].describe()

float64


count     5030.000000
mean     22185.089463
std      15205.699563
min       4000.000000
25%      11000.000000
50%      16000.000000
75%      32000.000000
max      54000.000000
Name: harga_satuan, dtype: float64

### 1.2 Kuantitas negatif → positif

In [5]:
print('Jumlah negatif sebelum:', (df['jumlah'] < 0).sum())
df.loc[df['jumlah'] < 0, 'jumlah'] = df.loc[df['jumlah'] < 0, 'jumlah'].abs()
print('Jumlah negatif sesudah :', (df['jumlah'] < 0).sum())

Jumlah negatif sebelum: 41
Jumlah negatif sesudah : 0


### 1.3 Imputasi kode_kasir kosong

In [6]:
print('NaN sebelum:', df['kode_kasir'].isna().sum())
df.loc[df['kode_kasir'].isna(), 'kode_kasir'] = 'K_UNKNOWN'
print('NaN sesudah :', df['kode_kasir'].isna().sum())

NaN sebelum: 261
NaN sesudah : 0


### 1.4 Slicing dengan `.iloc[]`

In [7]:
df.iloc[:5, [0, 1, 2, 5]]   # id_transaksi, tanggal, kode_gerai, jumlah

,id_transaksi,tanggal,kode_gerai,jumlah
0,TRX0241065,2025-06-02,G09,4
1,TRX0240510,2024-12-28,G09,1
2,TRX0244483,2024-09-04,G11,3
3,TRX0242532,2025-07-30,G05,2
4,TRX0243509,26/03/2025,G05,1


# Task 2 — Operasi Aritmatika (Feature Engineering)

In [8]:
df['Total_Kotor'] = df['harga_satuan'] * df['jumlah']
df['Nominal_Diskon'] = df['Total_Kotor'] * (df['diskon_persen'] / 100)
df['Total_Bersih'] = (df['Total_Kotor'] - df['Nominal_Diskon']) * 1.11   # + PPN 11%

df[['harga_satuan', 'jumlah', 'diskon_persen', 'Total_Kotor', 'Nominal_Diskon', 'Total_Bersih']].head()

,harga_satuan,jumlah,diskon_persen,Total_Kotor,Nominal_Diskon,Total_Bersih
0,13000.0,4,0,52000.0,0.0,57720.0
1,21000.0,1,0,21000.0,0.0,23310.0
2,7500.0,3,0,22500.0,0.0,24975.0
3,6500.0,2,0,13000.0,0.0,14430.0
4,6000.0,1,0,6000.0,0.0,6660.0


# Task 3 — Filtering

### 3.1 Multi-kondisi

In [9]:
filter_multi = df[
    (df['status'] == 'SELESAI') &
    (df['kode_gerai'].isin(['G05', 'G09'])) &
    (df['Total_Bersih'] > 100_000)
]
print('Jumlah baris memenuhi kriteria:', len(filter_multi))
filter_multi.head()

Jumlah baris memenuhi kriteria: 140


,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
7,TRX0241251,2024-10-09,G09,K035,SGR1204,4,47500.0,10,SELESAI,190000.0,19000.0,189810.0
24,TRX0244253,2024-07-18,G09,K026,BYI1502,4,30500.0,0,SELESAI,122000.0,0.0,135420.0
25,TRX0244999,2024-08-09,G05,K015,MIN1003,4,28500.0,0,SELESAI,114000.0,0.0,126540.0
90,TRX0240898,2025-12-09,G05,K010,BYI1509,3,46500.0,0,SELESAI,139500.0,0.0,154845.0
206,TRX0241868,2025-07-01,G05,K009,SGR1203,6,20500.0,0,SELESAI,123000.0,0.0,136530.0


### 3.2 `.query()` — Gerai G10, jumlah ≥ 5

In [10]:
filter_g10 = df.query("kode_gerai == 'G10' and jumlah >= 5")
print('Jumlah baris:', len(filter_g10))
filter_g10.head()

Jumlah baris: 39


,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
83,TRX0243880,2024-08-04,G10,K002,MKN1101,12,45500.0,0,SELESAI,546000.0,0.0,606060.0
166,TRX0241592,2024-05-11,G10,K033,SGR1202,6,13000.0,0,SELESAI,78000.0,0.0,86580.0
351,TRX0242686,2024-02-03,G10,K016,BYI1502,12,26000.0,0,SELESAI,312000.0,0.0,346320.0
556,TRX0241396,2025-12-16,G10,K009,RMT1406,12,5500.0,5,SELESAI,66000.0,3300.0,69597.0
879,TRX0243559,2025-09-30,G10,K016,SGR1206,6,47000.0,0,SELESAI,282000.0,0.0,313020.0


# Task 4 — Sorting

### 4.1 Multi-column sorting

In [11]:
df_sorted = df.sort_values(by=['kode_gerai', 'Total_Bersih'], ascending=[True, False])
df_sorted.head(10)

,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
244,TRX0243257,2024-11-24,G01,K033,MKN1102,1456,32500.0,20,SELESAI,47320000.0,9464000.0,42020160.0
2956,TRX0241367,2025-05-15,G01,K006,RMT1408,12,50500.0,0,SELESAI,606000.0,0.0,672660.0
3588,TRX0240608,2025-07-25,G01,K006,MKN1101,12,47000.0,0,SELESAI,564000.0,0.0,626040.0
2937,TRX0240631,2024-12-24,G01,K013,RMT1408,12,42000.0,0,SELESAI,504000.0,0.0,559440.0
5,TRX0240658,2025-07-18,G01,K008,RMT1408,12,45500.0,10,SELESAI,546000.0,54600.0,545454.0
3889,TRX0241537,2024-10-19,G01,K013,MKN1102,12,27000.0,0,SELESAI,324000.0,0.0,359640.0
2179,TRX0244635,2025-08-30,G01,K009,BYI1503,6,52500.0,0,SELESAI,315000.0,0.0,349650.0
4270,TRX0240428,2024-03-31,G01,K004,RMT1408,6,48500.0,0,SELESAI,291000.0,0.0,323010.0
695,TRX0244706,2024-08-23,G01,K013,SGR1204,6,47000.0,0,BATAL,282000.0,0.0,313020.0
4404,TRX0242179,2024-02-11,G01,K019,RMT1408,6,47000.0,0,SELESAI,282000.0,0.0,313020.0


### 4.2 Top-5 dengan `.nlargest()`

In [12]:
df.nlargest(5, 'Total_Bersih')

,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
994,TRX0241002,2024-03-18,G11,K038,MKN1101,2093,48000.0,10,SELESAI,100464000.0,10046400.0,100363536.0
4985,TRX0244221,2024-12-09,G09,K005,BYI1500,1301,51500.0,10,SELESAI,67001500.0,6700150.0,66934498.5
467,TRX0243696,2024-02-27,G06,K_UNKNOWN,SGR1201,892,54000.0,0,SELESAI,48168000.0,0.0,53466480.0
3393,TRX0241029,2025-05-12,G07,K028,SGR1205,1983,23000.0,10,SELESAI,45609000.0,4560900.0,45563391.0
244,TRX0243257,2024-11-24,G01,K033,MKN1102,1456,32500.0,20,SELESAI,47320000.0,9464000.0,42020160.0


# Task 5 — Grouping & Agregasi

### 5.1 Laporan agregasi per gerai (status SELESAI)

In [13]:
df_selesai = df[df['status'] == 'SELESAI']

laporan_gerai = (
    df_selesai.groupby('kode_gerai')
    .agg(
        Jumlah_Transaksi=('id_transaksi', 'count'),
        Total_Unit_Terjual=('jumlah', 'sum'),
        Total_Omzet_Bersih=('Total_Bersih', 'sum'),
        Rata_Rata_Transaksi=('Total_Bersih', 'mean'),
    )
    .sort_values('Total_Omzet_Bersih', ascending=False)
    .round(2)
)
laporan_gerai

,Jumlah_Transaksi,Total_Unit_Terjual,Total_Omzet_Bersih,Rata_Rata_Transaksi
kode_gerai,,,,
G11,398,3170,1.256382e+08,315673.75
G09,379,3027,1.011576e+08,266906.68
G06,383,3103,9.122862e+07,238194.84
G07,384,4284,8.055975e+07,209791.01
G01,450,2635,7.096738e+07,157705.28
G03,372,3274,5.999123e+07,161266.74
G10,401,1742,5.896489e+07,147044.62
G08,408,1700,3.563008e+07,87328.64
G12,406,1010,2.469922e+07,60835.52


### 5.2 Window function dengan `.groupby().transform()`

In [14]:
df['Kontribusi_Gerai_Persen'] = (
    df['Total_Bersih'] / df.groupby('kode_gerai')['Total_Bersih'].transform('sum')
) * 100

df[['id_transaksi', 'kode_gerai', 'Total_Bersih', 'Kontribusi_Gerai_Persen']].head()

,id_transaksi,kode_gerai,Total_Bersih,Kontribusi_Gerai_Persen
0,TRX0241065,G09,57720.0,0.056292
1,TRX0240510,G09,23310.0,0.022733
2,TRX0244483,G11,24975.0,0.018326
3,TRX0242532,G05,14430.0,0.060075
4,TRX0243509,G05,6660.0,0.027727


In [15]:
# Validasi: total kontribusi tiap gerai harus 100%
df.groupby('kode_gerai')['Kontribusi_Gerai_Persen'].sum().round(2)

kode_gerai
G01    100.0
G02    100.0
G03    100.0
G04    100.0
G05    100.0
G06    100.0
G07    100.0
G08    100.0
G09    100.0
G10    100.0
G11    100.0
G12    100.0
Name: Kontribusi_Gerai_Persen, dtype: float64